In [9]:
import ee
import earthaccess
import os
import numpy as np
import pandas as pd
import glob
import matplotlib.pyplot as plt

# Initialize Earth Engine
ee.Authenticate()
ee.Initialize()

# Configuration from your .ipynb

BBOX = [8.8, 55.8, 9.4, 56.1]; 

PATCH_SIZE = 13
START_DATE = "2014-05-01"
END_DATE = "2020-01-01"
PATCH_CENTER = PATCH_SIZE // 2
AOI = ee.Geometry.Rectangle(BBOX)

In [10]:
def load_ismn(file_path):
    with open(file_path, 'r') as f:
        header = f.readline().strip().split()

    try:
        network, station = header[1], header[2]
        lat, lon = float(header[3]), float(header[4])
    except (IndexError, ValueError):
        return None

    df = pd.read_csv(
        file_path, 
        sep=r'\s+', 
        skiprows=1, 
        header=None, 
        engine='python'
    )

    if df.shape[1] < 4:
        return None

    df = df.iloc[:, :4]
    df.columns = ['date', 'time', 'sm', 'flag']

    # Filter for Good quality
    df = df[df['flag'] == 'G']
    if df.empty: return None

    # Cleaning
    df['timestamp'] = pd.to_datetime(df['date'] + ' ' + df['time'])
    df['sm'] = pd.to_numeric(df['sm'], errors='coerce')
    df = df.dropna(subset=['sm'])

    # Unit conversion (Percentage to m3/m3)
    if df['sm'].mean() > 1.0:
        df['sm'] = df['sm'] / 100.0

    df['point_id'] = f"{network}_{station}"
    df['latitude'] = lat
    df['longitude'] = lon

    return df[['point_id', 'timestamp', 'sm', 'latitude', 'longitude']]

# --- EXECUTION ---
stm_files = glob.glob('Data/Denmark/HOBE/*/*.stm', recursive=True)

dfs = [load_ismn(f) for f in stm_files]
dfs = [d for d in dfs if d is not None]
df_combined = pd.concat(dfs, ignore_index=True)

# 1. MULTI-SENSOR COLLAPSE (Handle A, B, C duplicates)
# We group by timestamp to average the redundant sensors at each specific hour/minute
ground_truth = df_combined.groupby(
    ['point_id', 'timestamp', 'latitude', 'longitude'], as_index=False
)['sm'].mean()

# 2. DAILY AGGREGATION
# Now we move from hourly/sub-hourly to daily
ground_truth['timestamp'] = (
    ground_truth['timestamp']
    .dt.tz_localize(
        'Europe/Copenhagen',
        nonexistent='shift_forward',
        ambiguous='NaT'
    )
    .dropna()
    .dt.tz_convert('UTC')
    .dt.floor('D')
)

ground_truth = ground_truth.groupby(
    ['point_id', 'timestamp', 'latitude', 'longitude'], as_index=False
)['sm'].mean()
print(f"Final ground truth shape: {ground_truth.shape}")
print(f"Unique stations: {ground_truth['point_id'].nunique()}")

Final ground truth shape: (61284, 5)
Unique stations: 30


In [11]:
# ================= CONFIG =================
TARGET_SCALE = 1000
TARGET_CRS = 'EPSG:32632'
PATCH_RADIUS = (PATCH_SIZE - 1) // 2


# ================= DATA LOADERS =================

def get_ndvi(aoi, start, end):
    col = ee.ImageCollection("MODIS/061/MOD13A1") \
        .filterBounds(aoi) \
        .filterDate(start, end)

    def proc(img):
        ndvi = img.select("NDVI").multiply(0.0001).rename("NDVI")
        evi = img.select("EVI").multiply(0.0001).rename("EVI")

        return ee.Image.cat([ndvi, evi]) \
            .copyProperties(img, ["system:time_start"])

    return col.map(proc)


def get_s1(aoi, start, end):
    s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(aoi) \
        .filterDate(start, end) \
        .filter(ee.Filter.eq('instrumentMode', 'IW'))

    def proc(img):
        bands = img.bandNames()

        vv = ee.Image(
            ee.Algorithms.If(bands.contains('VV'), img.select('VV'), ee.Image.constant(-20))
        )

        vh = ee.Image(
            ee.Algorithms.If(bands.contains('VH'), img.select('VH'), vv)
        )

        angle = img.select('angle')  # always present in GRD

        vv = vv.toFloat().unmask(-20)
        vh = vh.toFloat().unmask(-20)
        angle = angle.toFloat().unmask(35)  # typical mid-angle fallback

        ratio = vv.subtract(vh).rename('VV_VH_Ratio')

        return ee.Image.cat([
            vv.rename('VV'),
            vh.rename('VH'),
            ratio,
            angle.rename('angle')
        ]).copyProperties(img, ['system:time_start'])

    return s1.map(proc)


def get_lst(aoi, start, end):
    return ee.ImageCollection('MODIS/061/MOD11A1') \
        .filterBounds(aoi).filterDate(start, end) \
        .map(lambda img: img.select('LST_Day_1km')
             .multiply(0.02).subtract(273.15)
             .rename('LST')
             .copyProperties(img, ['system:time_start']))

def add_vpd(img):
    img = ee.Image(img)
    T = img.select("T_air")
    Td = img.select("Td_air")

    es = T.expression(
        "0.6108 * exp((17.27 * T) / (T + 237.3))",
        {"T": T}
    )

    ea = Td.expression(
        "0.6108 * exp((17.27 * Td) / (Td + 237.3))",
        {"Td": Td}
    )

    vpd = es.subtract(ea).rename("VPD")

    return img.addBands(vpd)

def compute_api(col, date, alpha=0.85, days=10):
    start = ee.Date(date).advance(-days, "day")
    subset = col.filterDate(start, date).select("precip").sort("system:time_start")

    size = subset.size()

    def compute():
        imgs = subset.toList(size)

        def body(i, acc):
            img = ee.Image(imgs.get(i))
            w = ee.Number(alpha).pow(i)
            return ee.Image(acc).add(img.multiply(w))

        return ee.Image(
            ee.List.sequence(0, size.subtract(1))
              .iterate(body, ee.Image.constant(0))
        )

    return ee.Image(
        ee.Algorithms.If(
            size.gt(0),
            compute(),
            ee.Image.constant(0).rename("API")
        )
    )


def get_era5(aoi, start, end):
    col = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
        .filterBounds(aoi).filterDate(start, end)

    def proc(img):
        img = ee.Image(img)  # FORCE CAST (critical fix)

        base = ee.Image.cat([
            img.select("volumetric_soil_water_layer_1").rename("ERA5_SM"),
            img.select("total_precipitation_sum").rename("precip"),
            img.select("surface_net_solar_radiation_sum").divide(1e6).rename("ssr"),
            img.select("temperature_2m").subtract(273.15).rename("T_air"),
            img.select("dewpoint_temperature_2m").subtract(273.15).rename("Td_air")
        ]).copyProperties(img, ["system:time_start"])

        return add_vpd(base)

    return col.map(proc)


def get_lai(aoi, start, end):
    return ee.ImageCollection("MODIS/061/MCD15A3H") \
        .filterBounds(aoi).filterDate(start, end) \
        .map(lambda img: img.select("Lai").multiply(0.1)
             .rename("LAI")
             .copyProperties(img, ['system:time_start']))


def get_topo():
    dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')
    slope = ee.Terrain.slope(dem).rename('slope')
    aspect = ee.Terrain.aspect(dem).rename('aspect')
    return ee.Image.cat([dem.rename('elevation'), slope, aspect])


def get_soil():
    return ee.Image.cat([
        ee.Image("projects/soilgrids-isric/clay_mean")
            .select("clay_0-5cm_mean").rename("clay"),
        ee.Image("projects/soilgrids-isric/sand_mean")
            .select("sand_0-5cm_mean").rename("sand"),
        ee.Image("projects/soilgrids-isric/silt_mean")
            .select("silt_0-5cm_mean").rename("silt")
    ])


# ================= SAFE TEMPORAL FUNCTIONS =================

def safe_median(col, bands):
    return ee.Image(
        ee.Algorithms.If(
            col.size().gt(0),
            col.median(),
            ee.Image.constant([0]*len(bands)).rename(bands)
        )
    )


def get_window(col, date, bands, days):
    start = ee.Date(date).advance(-days, 'day')
    subset = col.filterDate(start, date)  # STRICTLY PAST

    img = ee.Image(
        ee.Algorithms.If(
            subset.size().gt(0),
            subset.median(),
            safe_median(col, bands)
        )
    )

    return img.select(bands)


def get_daily(col, date, bands):
    subset = col.filterDate(date, ee.Date(date).advance(1, 'day'))

    return ee.Image(
        ee.Algorithms.If(
            subset.size().gt(0),
            subset.median(),
            ee.Image.constant([0]*len(bands)).rename(bands)
        )
    )


# ================= FEATURE BUILDER =================

def get_era5_window(col, date):
    start = ee.Date(date).advance(-6, 'day')
    end = ee.Date(date).advance(1, 'day')

    subset = col.filterDate(start, end)

    base = ee.Image(
        ee.Algorithms.If(
            subset.size().gt(0),
            subset.mean(),
            ee.Image.constant([0]*5).rename(
                ['ERA5_SM','precip','ssr','T_air','Td_air']
            )
        )
    )

    base = add_vpd(base)

    api = compute_api(col, date)

    return ee.Image.cat([
        base,
        api.rename("API")
    ])


def build_stack(date):
    date = ee.Date(date)

    era5 = get_era5_window(era5_col, date)

    dynamic = ee.Image.cat([
        get_window(ndvi_col, date, ['NDVI', 'EVI'], 30),
        get_window(lai_col, date, ['LAI'], 30),
        get_window(s1_col, date, ['VV','VH','VV_VH_Ratio', 'angle'], 12),
        get_window(lst_col, date, ['LST'], 7),
        era5
    ]).resample('bilinear')

    img = ee.Image.cat([
        dynamic,
        soil_img,
        topo_img
    ]).reproject(
        crs=TARGET_CRS,
        scale=TARGET_SCALE
    )

    kernel = ee.Kernel.square(radius=PATCH_RADIUS)
    img = img.neighborhoodToArray(kernel)

    return img.set('system:time_start', date.millis())


# ================= EXECUTION =================

BUFFER_START = ee.Date(START_DATE).advance(-30, 'day')

ndvi_col = get_ndvi(AOI, BUFFER_START, END_DATE)
s1_col = get_s1(AOI, BUFFER_START, END_DATE)
lai_col = get_lai(AOI, BUFFER_START, END_DATE)
lst_col = get_lst(AOI, START_DATE, END_DATE)
era5_col = get_era5(AOI, START_DATE, END_DATE)

soil_img = get_soil()
topo_img = get_topo()

dates = ee.List.sequence(
    0,
    ee.Date(END_DATE).difference(ee.Date(START_DATE), 'day').subtract(1)
).map(lambda d: ee.Date(START_DATE).advance(d, 'day'))


points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([r.longitude, r.latitude]), {'point_id': r.point_id})
    for r in ground_truth[['point_id','longitude','latitude']].drop_duplicates().itertuples()
])


collection = ee.ImageCollection(dates.map(build_stack))


def sample(img):
    ts = img.get('system:time_start')

    return img.sampleRegions(
        collection=points,
        properties=['point_id'],
        scale=TARGET_SCALE,   # MATCH projection
        tileScale=8,
        geometries=False
    ).map(lambda f: f.set('timestamp', ts))


dataset = collection.map(sample).flatten()


task = ee.batch.Export.table.toDrive(
    collection=dataset,
    description='ML_ready_1km_patches',
    fileFormat='TFRecord'
)

task.start()
print("Export started")

Export started


In [12]:
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
from tfrecord.reader import tfrecord_loader
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

GLOBAL_SEED = 42
REPRO_MODE = True  # Set True for deterministic/repeatable results

def seed_everything(seed=GLOBAL_SEED, deterministic=REPRO_MODE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        # Needed for stricter CUDA reproducibility in some ops.
        os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True, warn_only=True)
    else:
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False
        torch.use_deterministic_algorithms(False)


seed_everything()

# 1. Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"REPRO_MODE={REPRO_MODE}, GLOBAL_SEED={GLOBAL_SEED}")

Using device: cuda
REPRO_MODE=True, GLOBAL_SEED=42


In [13]:
class HOBEPatchDataset(IterableDataset):
    def __init__(self, data_path, features):
        self.data_path = data_path
        self.features = features
        self.description = {f: "float" for f in features}
        self.description['point_id'] = "byte"
        self.description['timestamp'] = "float"

    def __iter__(self):
        loader = tfrecord_loader(self.data_path, None, self.description, compression_type='gzip')
        for record in loader:
            try:
                patch_list = []
                for feat in self.features:
                    arr = np.array(record[feat])
                    if arr.size != 169:
                        continue  # skip corrupted
                    patch_list.append(arr.reshape(13, 13))

                if len(patch_list) != len(self.features):
                    continue

                x = torch.from_numpy(np.stack(patch_list)).float()
                pid = record['point_id'].decode('utf-8')
                ts = int(record['timestamp'].item())

                yield x, pid, ts

            except:
                continue

In [14]:
features = [
    'NDVI', 'EVI', 'LAI', 'VV', 'VH', 'VV_VH_Ratio', 'angle',
    'LST', 'ERA5_SM', 'precip', 'ssr', 'T_air', 'Td_air', 'VPD', 'API',
    'clay', 'sand', 'silt', 'elevation', 'slope', 'aspect'
 ]
FEATURE_INDEX = {f: i for i, f in enumerate(features)}

def get_anchor(x, i, anchor_type):
    if anchor_type != 'ERA5':
        raise ValueError("Only ERA5 anchor is supported")

    return x[i, FEATURE_INDEX['ERA5_SM']].mean().item()

# Standardize dictionary keys to (str, int_ms)
gt_lookup = {
    (str(row.point_id), int(row.timestamp.value // 10**6)): row.sm 
    for row in ground_truth.itertuples()
}

ds = HOBEPatchDataset("Data/Denmark/ML_ready_1km_patches.tfrecord.gz", features)
loader = DataLoader(ds, batch_size=64, num_workers=0)

print("Caching data to RAM...")
cached_data = []
for x, pids, tss in loader:
    cached_data.append((x, pids, tss))

Caching data to RAM...


In [15]:
all_pids = np.unique([p for _, pids, _ in cached_data for p in pids])

np.random.seed(42)
np.random.shuffle(all_pids)

train_stations = set(all_pids[:int(0.8 * len(all_pids))])
test_stations = set(all_pids[int(0.8 * len(all_pids)):])

In [16]:
def normalize_tensor(x):
    # x: [Batch, Channels, H, W]
    # We normalize each channel independently across the batch
    for c in range(x.shape[1]):
        c_slice = x[:, c, :, :]
        c_min = c_slice.min()
        c_max = c_slice.max()
        
        # If the channel isn't "dead" (max != min)
        if c_max > c_min:
            x[:, c, :, :] = (c_slice - c_min) / (c_max - c_min)
        else:
            # If it's a dead channel (like your 3, 4, 5, 8), keep it at 0
            x[:, c, :, :] = 0.0
    return x


In [17]:
import gc
import math

BASE_CHANNELS = len(features)

def add_temporal_features(cached_list, base_channels=BASE_CHANNELS):
    print("Injecting Day_Sin and Day_Cos features (in-place)...")

    for idx, (x, pids, tss) in enumerate(cached_list):
        if x.shape[1] < base_channels:
            raise ValueError(f"Expected at least {base_channels} channels, got {x.shape[1]}")

        # Always rebuild from original base channels so reruns do not duplicate temporal bands.
        x_base = x[:, :base_channels, :, :]
        sin_list = []
        cos_list = []

        for ts in tss:
            dt = pd.to_datetime(ts.item(), unit='ms')
            doy = dt.dayofyear

            sin_val = math.sin(2 * math.pi * doy / 365.25)
            cos_val = math.cos(2 * math.pi * doy / 365.25)

            sin_patch = torch.full((1, 13, 13), sin_val, dtype=x_base.dtype, device=x_base.device)
            cos_patch = torch.full((1, 13, 13), cos_val, dtype=x_base.dtype, device=x_base.device)

            sin_list.append(sin_patch)
            cos_list.append(cos_patch)

        sin_tensor = torch.stack(sin_list)
        cos_tensor = torch.stack(cos_list)

        # Final shape is always [Batch, base_channels + 2, 13, 13].
        x_new = torch.cat([x_base, sin_tensor, cos_tensor], dim=1)

        # In-place replacement avoids keeping both old and new full cached lists in memory.
        cached_list[idx] = (x_new, pids, tss)

    gc.collect()
    return cached_list

def get_normalization_stats_from_cache(cached_list):
    all_samples = [x for x, _, _ in cached_list]
    stacked = torch.cat(all_samples, dim=0)
    means = stacked.mean(dim=(0, 2, 3))
    stds = stacked.std(dim=(0, 2, 3)).clamp_min(1e-6)
    return means, stds

cached_data = add_temporal_features(cached_data)

Injecting Day_Sin and Day_Cos features (in-place)...


In [18]:
from collections import deque
import gc

def add_precip_features(cached_list):
    print("Adding streaming precip accumulation features...")

    # Per-station rolling state to avoid flattening all samples into RAM.
    station_state = {}

    for idx, (x, pids, tss) in enumerate(cached_list):
        p3_list = []
        p7_list = []

        for i in range(len(pids)):
            pid = pids[i]
            ts = int(tss[i].item())
            curr = x[i, FEATURE_INDEX['precip']]

            state = station_state.get(pid)
            if state is None:
                state = {
                    'last_ts': None,
                    'q3': deque(),
                    'q7': deque(),
                    'sum3': torch.zeros_like(curr),
                    'sum7': torch.zeros_like(curr),
                }
                station_state[pid] = state

            # If records for a station arrive out of order, reset to avoid leakage.
            if state['last_ts'] is not None and ts < state['last_ts']:
                state['q3'].clear()
                state['q7'].clear()
                state['sum3'].zero_()
                state['sum7'].zero_()

            state['last_ts'] = ts

            if len(state['q3']) == 3:
                state['sum3'] -= state['q3'].popleft()
            state['q3'].append(curr)
            state['sum3'] += curr

            if len(state['q7']) == 7:
                state['sum7'] -= state['q7'].popleft()
            state['q7'].append(curr)
            state['sum7'] += curr

            p3_list.append(state['sum3'].clone())
            p7_list.append(state['sum7'].clone())

        p3 = torch.stack(p3_list, dim=0).unsqueeze(1)
        p7 = torch.stack(p7_list, dim=0).unsqueeze(1)
        x_new = torch.cat([x, p3, p7], dim=1)

        # In-place replacement avoids doubling the cached dataset in memory.
        cached_list[idx] = (x_new, pids, tss)

    gc.collect()
    return cached_list

cached_data = add_precip_features(cached_data)

Adding streaming precip accumulation features...


In [19]:
def get_train_stats(cached_list):
    xs = []
    for x, pids, _ in cached_list:
        mask = [i for i, p in enumerate(pids) if p in train_stations]
        if mask:
            xs.append(x[mask])

    stacked = torch.cat(xs, dim=0)
    means = stacked.mean(dim=(0, 2, 3))
    stds = stacked.std(dim=(0, 2, 3)).clamp_min(1e-6)
    return means, stds


def apply_zscore_normalization(cached_list, means, stds):
    means_b = means.view(1, -1, 1, 1)
    stds_b = stds.view(1, -1, 1, 1)

    for idx, (x, pids, tss) in enumerate(cached_list):
        x_norm = ((x - means_b) / (stds_b + 1e-6)).to(torch.float32)
        cached_list[idx] = (x_norm, pids, tss)

    return cached_list


means, stds = get_train_stats(cached_data)
cached_data = apply_zscore_normalization(cached_data, means, stds)

print(f"Applied z-score normalization to all channels. Channel count: {means.numel()}")

Applied z-score normalization to all channels. Channel count: 25


In [20]:
FINAL_CHANNELS = cached_data[0][0].shape[1]
print("Final channels:", FINAL_CHANNELS)

Final channels: 25


In [21]:
class SoilCNN(nn.Module):
    def __init__(self, in_channels=25):
        super(SoilCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32) # Added
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64) # Added
        self.drop = nn.Dropout2d(0.3) 
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=0) 
        self.pool = nn.AdaptiveAvgPool2d((3, 3))
        self.fc1 = nn.Linear(128 * 3 * 3, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.drop(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = SoilCNN(in_channels=FINAL_CHANNELS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4) 
criterion = nn.SmoothL1Loss(beta=0.02)

In [22]:
# Fast residual check: O(total_samples) instead of nested O(gt * total_samples)
era5_by_key = {}

for x, pids, tss in cached_data:
    for i in range(len(pids)):
        floored_ts = int((tss[i].item() // 86400000) * 86400000)
        key = (pids[i], floored_ts)
        era5_by_key[key] = x[i, FEATURE_INDEX['ERA5_SM'], PATCH_CENTER, PATCH_CENTER].item()

residuals = [
    sm - era5_by_key[key]
    for key, sm in gt_lookup.items()
    if key in era5_by_key
]

print("Residual stats:")
if residuals:
    arr = np.asarray(residuals, dtype=np.float32)
    print(np.mean(arr), np.std(arr), np.min(arr), np.max(arr))
    print(f"Matched pairs: {arr.size}")
else:
    print("No matched (point_id, timestamp) pairs found.")

Residual stats:
0.22524098 1.0182879 -1.3703165 4.240639
Matched pairs: 25824


In [23]:
def build_supervised_batches(cached_list, station_set, target_chunk_size=1024):
    """
    Build (x, y_residual) batches once to remove repeated Python dict/set work each epoch.
    Returns CPU tensors; training loop transfers each chunk to GPU.
    """
    chunks = []
    x_buf, y_buf = [], []

    era5_idx = FEATURE_INDEX['ERA5_SM']
    era5_mu = means[era5_idx].item()
    era5_sigma = stds[era5_idx].item()

    for x, pids, tss in cached_list:
        floored = ((tss // 86400000) * 86400000).tolist()

        # x is normalized; convert ERA5 center back to raw units for residual target.
        era5_center_raw = x[:, era5_idx, PATCH_CENTER, PATCH_CENTER] * era5_sigma + era5_mu

        keep_idx = []
        residual_targets = []

        for i, pid in enumerate(pids):
            if pid not in station_set:
                continue

            label = gt_lookup.get((pid, int(floored[i])))
            if label is None or np.isnan(label):
                continue

            keep_idx.append(i)
            residual_targets.append(label - era5_center_raw[i].item())

        if keep_idx:
            x_sel = x[keep_idx].contiguous()
            y_sel = torch.tensor(residual_targets, dtype=torch.float32).unsqueeze(1)
            x_buf.append(x_sel)
            y_buf.append(y_sel)

            total = sum(t.shape[0] for t in x_buf)
            if total >= target_chunk_size:
                chunks.append((torch.cat(x_buf, dim=0), torch.cat(y_buf, dim=0)))
                x_buf, y_buf = [], []

    if x_buf:
        chunks.append((torch.cat(x_buf, dim=0), torch.cat(y_buf, dim=0)))

    return chunks


print(f"Starting Training: {len(train_stations)} train, {len(test_stations)} test stations.")

train_batches = build_supervised_batches(cached_data, train_stations, target_chunk_size=1024)
print(f"Prepared {len(train_batches)} train chunks")

amp_enabled = (device.type == 'cuda') and (not REPRO_MODE)
if amp_enabled:
    scaler = torch.amp.GradScaler('cuda')
else:
    scaler = None

for epoch in range(15):
    model.train()
    total_loss, match_count = 0.0, 0

    for x_cpu, y_cpu in train_batches:
        x_batch = x_cpu.to(device, non_blocking=True)
        y_batch = y_cpu.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp_enabled):
            pred_residual = model(x_batch)
            loss = criterion(pred_residual, y_batch)

        if amp_enabled:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        bs = y_batch.shape[0]
        total_loss += loss.item() * bs
        match_count += bs

    if match_count == 0:
        print(f"Epoch {epoch} | No matched training samples")
    else:
        print(f"Epoch {epoch} | Train Loss: {total_loss / match_count:.6f}")

Starting Training: 24 train, 6 test stations.


Prepared 23 train chunks


/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/torch/autograd/graph.py:869: UserWarning: adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 0 | Train Loss: 0.060788
Epoch 1 | Train Loss: 0.042624
Epoch 2 | Train Loss: 0.039800
Epoch 3 | Train Loss: 0.038021
Epoch 4 | Train Loss: 0.037983
Epoch 5 | Train Loss: 0.039798
Epoch 6 | Train Loss: 0.040360
Epoch 7 | Train Loss: 0.037136
Epoch 8 | Train Loss: 0.034130
Epoch 9 | Train Loss: 0.031702
Epoch 10 | Train Loss: 0.029890
Epoch 11 | Train Loss: 0.028549
Epoch 12 | Train Loss: 0.027706
Epoch 13 | Train Loss: 0.027078
Epoch 14 | Train Loss: 0.026523


In [24]:
def build_eval_batches(cached_list, station_set, target_chunk_size=2048):
    """
    Build (x, y_true, era5_center_raw) batches for fast batched GPU inference.
    """
    chunks = []
    x_buf, y_buf, era5_buf = [], [], []

    era5_idx = FEATURE_INDEX['ERA5_SM']
    era5_mu = means[era5_idx].item()
    era5_sigma = stds[era5_idx].item()

    for x, pids, tss in cached_list:
        floored = ((tss // 86400000) * 86400000).tolist()

        # x is normalized; convert ERA5 center back to raw units for final prediction.
        era5_center_raw = x[:, era5_idx, PATCH_CENTER, PATCH_CENTER] * era5_sigma + era5_mu

        keep_idx = []
        y_true = []

        for i, pid in enumerate(pids):
            if pid not in station_set:
                continue

            label = gt_lookup.get((pid, int(floored[i])))
            if label is None or np.isnan(label):
                continue

            keep_idx.append(i)
            y_true.append(label)

        if keep_idx:
            x_sel = x[keep_idx].contiguous()
            y_sel = torch.tensor(y_true, dtype=torch.float32)
            era5_sel = era5_center_raw[keep_idx].to(torch.float32).contiguous()

            x_buf.append(x_sel)
            y_buf.append(y_sel)
            era5_buf.append(era5_sel)

            total = sum(t.shape[0] for t in x_buf)
            if total >= target_chunk_size:
                chunks.append((
                    torch.cat(x_buf, dim=0),
                    torch.cat(y_buf, dim=0),
                    torch.cat(era5_buf, dim=0)
                ))
                x_buf, y_buf, era5_buf = [], [], []

    if x_buf:
        chunks.append((
            torch.cat(x_buf, dim=0),
            torch.cat(y_buf, dim=0),
            torch.cat(era5_buf, dim=0)
        ))

    return chunks


model.eval()
preds, actuals = [], []

test_batches = build_eval_batches(cached_data, test_stations, target_chunk_size=2048)
print(f"Prepared {len(test_batches)} eval chunks")

amp_enabled = (device.type == 'cuda') and (not REPRO_MODE)

with torch.no_grad():
    for x_cpu, y_cpu, era5_cpu in test_batches:
        x_input = x_cpu.to(device, non_blocking=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp_enabled):
            residual = model(x_input).squeeze(1)

        pred = residual.float().cpu() + era5_cpu

        preds.extend(pred.tolist())
        actuals.extend(y_cpu.tolist())

if len(actuals) == 0:
    print("Unseen Station R2 Score: no matched samples")
else:
    print(f"Unseen Station R2 Score: {r2_score(actuals, preds):.4f}")

Prepared 5 eval chunks
Unseen Station R2 Score: 0.2682


In [25]:
print("\n=== CHANNEL STATS ===")
sample_x, _, _ = cached_data[0]

for i in range(sample_x.shape[1]):
    mean = sample_x[:, i].mean().item()
    std = sample_x[:, i].std().item()
    print(f"Channel {i}: mean={mean:.3f}, std={std:.3f}")


=== CHANNEL STATS ===
Channel 0: mean=0.445, std=0.051
Channel 1: mean=0.360, std=0.049
Channel 2: mean=0.394, std=0.091
Channel 3: mean=0.371, std=0.368
Channel 4: mean=-0.912, std=0.092
Channel 5: mean=0.412, std=0.409
Channel 6: mean=-0.179, std=0.220
Channel 7: mean=0.401, std=0.213
Channel 8: mean=-1.718, std=0.053
Channel 9: mean=-1.063, std=0.159
Channel 10: mean=1.408, std=0.106
Channel 11: mean=-0.342, std=0.019
Channel 12: mean=-1.305, std=0.091
Channel 13: mean=1.100, std=0.113
Channel 14: mean=-1.368, std=0.055
Channel 15: mean=0.018, std=0.970
Channel 16: mean=0.015, std=0.977
Channel 17: mean=-0.040, std=0.989
Channel 18: mean=0.037, std=0.939
Channel 19: mean=-0.017, std=0.990
Channel 20: mean=0.014, std=0.980
Channel 21: mean=1.300, std=0.013
Channel 22: mean=-0.655, std=0.023
Channel 23: mean=-1.265, std=0.083
Channel 24: mean=-1.477, std=0.039


In [26]:
dead_channels = (stds < 1e-5).nonzero().flatten().tolist()
print("Dead channels:", dead_channels)

Dead channels: []


In [27]:
def check_tensor(x):
    if torch.isnan(x).any():
        print("NaNs detected")
    if torch.isinf(x).any():
        print("Infs detected")

for x, _, _ in cached_data[:5]:
    check_tensor(x)

In [28]:
vals = list(gt_lookup.values())
print("\nTarget stats:")
print(f"Mean: {np.mean(vals):.4f}, Std: {np.std(vals):.4f}")
print(f"Min: {np.min(vals):.4f}, Max: {np.max(vals):.4f}")


Target stats:
Mean: 0.2159, Std: 0.0808
Min: 0.0000, Max: 0.4940


In [29]:
print(f"Train stations: {len(train_stations)}")
print(f"Test stations: {len(test_stations)}")
print(f"Total samples: {sum(len(p) for _, p, _ in cached_data)}")

Train stations: 24
Test stations: 6
Total samples: 62130


In [30]:
print("\n=== FEATURE ↔ TARGET CORR ===")
for i, f in enumerate(features):
    vals = []
    targets = []
    for x, pids, tss in cached_data:
        for j in range(len(pids)):
            key = (pids[j], int((tss[j].item() // 86400000) * 86400000))
            if key in gt_lookup:
                vals.append(x[j, i, PATCH_CENTER, PATCH_CENTER].item())
                targets.append(gt_lookup[key])
    if len(vals) > 0:
        corr = np.corrcoef(vals, targets)[0,1]
        print(f"{f}: {corr:.3f}")


=== FEATURE ↔ TARGET CORR ===
NDVI: -0.314
EVI: -0.354
LAI: -0.380
VV: -0.020
VH: 0.056
VV_VH_Ratio: -0.018
angle: -0.034
LST: -0.406
ERA5_SM: 0.467
precip: 0.170
ssr: -0.386
T_air: -0.374
Td_air: -0.322
VPD: -0.437
API: 0.128
clay: 0.091
sand: -0.077
silt: 0.042
elevation: 0.065
slope: 0.176
aspect: 0.025
